# Figure 2 — A benchmark calibrated to real data (S08, S09, S19)

Sources: `benchmarks/real_reference/compare.csv` (S08; v1/v2 vs six real
datasets), `benchmarks/synthetic_multimodal/v3_calibration/compare_v3.csv` and
`gates_v3.json` (S09; adds v3 rows), `benchmarks/real_reference/distributions/*.csv`
(S08 per-cell/per-gene distributions), `benchmarks/dyngen/realism.csv` (S19).
Column meanings: RESULT_S08.md (`match` = distance ≤ max(real spread, fixed
tolerance) for every applicable real reference; `within_real_range` = synthetic
median inside the real min–max range).

In [1]:
from _style import *
import json as _json, re

prov = Provenance("Fig2", notebook="notebooks/fig2_benchmark.ipynb")
cmp8 = pd.read_csv(prov.use("benchmarks/real_reference/compare.csv", "a,c", "S08"))
cmp3 = pd.read_csv(prov.use("benchmarks/synthetic_multimodal/v3_calibration/compare_v3.csv", "a,b,c", "S09"))
gates3 = _json.loads(prov.use("benchmarks/synthetic_multimodal/v3_calibration/gates_v3.json", "c", "S09").read_text())
dyn = pd.read_csv(prov.use("benchmarks/dyngen/realism.csv", "c", "S19"))

# compare_v3.csv repeats the S08 v1/v2 rows; check they are identical before using it alone
key = ["synthetic", "statistic"]
m = cmp8.merge(cmp3, on=key, suffixes=("_8", "_3"))
assert len(m) == len(cmp8), "compare_v3 does not contain every S08 row"
assert (m["match_8"] == m["match_3"]).all()
prov.value("n_real_statistics_ids", len(set(re.match(r"^[A-Z]\d+", s).group(0) for s in cmp3.statistic)))

def gen_of(preset):
    return preset.split("_")[0] if preset != "v1" else "v1"

cmp3["generator"] = cmp3.preset.map(gen_of)
preset_order = ["v1", "v2_default", "v2_batch", "v2_stress", "v3_default", "v3_batch", "v3_stress", "v3_stationary"]

## Panel a — real vs synthetic values for six headline statistics

In [2]:
stats_a = [("C3_hvg_log_mean", "HVG log mean"), ("C5_zero_fraction", "zero fraction"),
           ("C7_participation_ratio", "participation ratio"), ("C4_dispersion", "dispersion φ"),
           ("S1_unspliced_fraction", "unspliced fraction"), ("G3_multi_time_clone_fraction", "multi-time clones")]

fig = plt.figure(figsize=(DOUBLE_COL, 7.2))
gs = fig.add_gridspec(4, 6, height_ratios=[1.0, 1.9, 0.62, 1.0], hspace=0.45, wspace=0.75, top=0.95, bottom=0.05)
axs_a = [fig.add_subplot(gs[0, i]) for i in range(6)]
for i, (ax, (st, lab)) in enumerate(zip(axs_a, stats_a)):
    sub = cmp3[cmp3.statistic == st]
    real = _json.loads(sub.real_values_or_medians.iloc[0].replace("NaN", "null"))
    real = {k: v for k, v in real.items() if v is not None}
    lo, hi = min(real.values()), max(real.values())
    ax.axhspan(lo, hi, color="#DDDDDD", lw=0)
    xs = {"real": 0, "v1": 1, "v2": 2, "v3": 3}
    ax.scatter([0] * len(real), list(real.values()), s=10, color=C_REAL, zorder=3)
    for g in ["v1", "v2", "v3"]:
        v = sub[sub.generator == g].synthetic_scalar_or_median.astype(float)
        jit = np.linspace(-0.18, 0.18, len(v)) if len(v) > 1 else [0]
        ax.scatter(xs[g] + np.asarray(jit), v, s=7, color=C_GEN[g], zorder=3, lw=0)
    ax.set_xticks([0, 1, 2, 3]); ax.set_xticklabels(["real", "v1", "v2", "v3"], rotation=55, fontsize=6)
    ax.set_title(lab, fontsize=6.5)
    ax.set_xlim(-0.6, 3.6)
    if i == 0:
        panel_letter(ax, "a", x=-0.55)
    prov.value(f"a_{st}_real_min", lo); prov.value(f"a_{st}_real_max", hi)
    for g in ["v1", "v2", "v3"]:
        v = sub[sub.generator == g].synthetic_scalar_or_median.astype(float)
        prov.value(f"a_{st}_{g}_median", float(np.nanmedian(v)))
prov.panel("a", "Six statistics: per-real-dataset value (black) and real range (grey band) vs each synthetic dataset (v1, v2 presets, v3 presets)")

## Panel b — within-real-range per statistic × preset (fraction of seeds)

In [3]:
axb = fig.add_subplot(gs[1, :4])
stat_order = sorted(cmp3.statistic.unique())
H = cmp3.assign(w=cmp3.within_real_range.map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0})) \
        .pivot_table(index="statistic", columns="preset", values="w", aggfunc="mean") \
        .reindex(index=stat_order, columns=preset_order)
cmap = matplotlib.colormaps["Greens"].copy(); cmap.set_bad("#BDBDBD")
im = axb.imshow(H.T.values, aspect="auto", cmap=cmap, vmin=0, vmax=1)
axb.set_yticks(range(len(preset_order))); axb.set_yticklabels(preset_order)
axb.set_xticks(range(len(stat_order))); axb.set_xticklabels(stat_order, rotation=90, fontsize=5.2)
cax = axb.inset_axes([1.025, 0.15, 0.022, 0.7]); cb = fig.colorbar(im, cax=cax, orientation="vertical")
cb.ax.tick_params(labelsize=6); cb.set_label("fraction of seeds\nwithin real range", fontsize=6)
axb.text(0.0, 1.04, "grey: not applicable", transform=axb.transAxes, ha="left", va="bottom", fontsize=5.5, color="#666666")
panel_letter(axb, "b", x=-0.1)
prov.panel("b", "Within-real-range indicator per statistic and preset (mean over seeds); grey = not applicable")
for p in preset_order:
    prov.value(f"b_within_range_rate_{p}", float(np.nanmean(H[p].values)))

## Panel c — S08 match rate per preset, and dyngen (S19)

In [4]:
axc = fig.add_subplot(gs[1, 5:])
mr = cmp3.assign(mt=cmp3.match.astype(str).eq("True")).groupby("preset").mt.mean().reindex(preset_order)
dyn["mt"] = dyn.match.astype(str).eq("True")
dmr = dyn.groupby("synthetic").mt.mean()
labels = preset_order + ["dyngen (9)"]
vals = list(mr.values) + [dyn.mt.mean()]
cols = [C_GEN[gen_of(p)] for p in preset_order] + [C_GEN["dyngen"]]
axc.barh(range(len(labels)), vals, color=cols, height=0.7)
axc.scatter(dmr.values, [len(labels) - 1] * len(dmr), s=6, color="black", zorder=3, lw=0)
for i, v in enumerate(vals):
    xt = max(v, dmr.max()) if i == len(vals) - 1 else v
    axc.text(xt + (0.06 if i == len(vals) - 1 else 0.015), i, f2(v), va="center", fontsize=6)
axc.set_yticks(range(len(labels))); axc.set_yticklabels(labels)
axc.invert_yaxis(); axc.set_xlim(0, 0.75); axc.set_xlabel("match rate (S08 rule)")
panel_letter(axc, "c", x=-1.1)
prov.panel("c", "Match rate (fraction of statistics matched) per preset; dyngen = mean over nine datasets (dots: per dataset)")
for p, v in mr.items():
    prov.value(f"c_match_rate_{p}", float(v))
prov.value("c_match_rate_dyngen_mean", float(dyn.mt.mean()))
prov.value("c_match_rate_dyngen_min", float(dmr.min())); prov.value("c_match_rate_dyngen_max", float(dmr.max()))
prov.value("c_n_statistics_by_preset", {p: int((cmp3.preset == p).sum()) for p in preset_order})

{'v1': 88,
 'v2_default': 88,
 'v2_batch': 89,
 'v2_stress': 87,
 'v3_default': 96,
 'v3_batch': 95,
 'v3_stress': 95,
 'v3_stationary': 90}

## Panel d — distributions (S08): zero fraction and HVG log mean, real vs v1/v2

In [5]:
dist_dir = "benchmarks/real_reference/distributions"
reals = ["scnt_hema", "larry", "setty", "pancreas", "dentate"]
real_cols = ["#000000", "#444444", "#777777", "#999999", "#BBBBBB"]
for j, (st, lab) in enumerate([("C5_zero_fraction", "per-gene zero fraction"), ("C3_hvg_log_mean", "HVG log mean"),
                               ("C4_dispersion", "dispersion")]):
    ax = fig.add_subplot(gs[3, 2 * j:2 * j + 2])
    for r, c in zip(reals, real_cols):
        p = resolve(f"{dist_dir}/{r}__{st}.csv")
        if p.exists():
            v = np.sort(pd.read_csv(prov.use(p, "d", "S08")).value.dropna().values)
            ax.plot(v, np.linspace(0, 1, len(v)), color=c, lw=1.0, label=r)
    for g, nm in [("v1", "v1_default_s0"), ("v2", "v2_default_s0")]:
        v = np.sort(pd.read_csv(prov.use(f"{dist_dir}/{nm}__{st}.csv", "d", "S08")).value.dropna().values)
        ax.plot(v, np.linspace(0, 1, len(v)), color=C_GEN[g], lw=1.2, ls="--", label=nm)
    ax.set_xlabel(lab); ax.set_ylabel("ECDF")
    if st in ("C3_hvg_log_mean", "C4_dispersion"):
        ax.set_xscale("log")
    if j == 0:
        panel_letter(ax, "d", x=-0.3)
    if j == 2:
        ax.legend(fontsize=6, loc="lower right", handlelength=1.8, borderaxespad=0.2)
prov.panel("d", "ECDFs of three S08 distributions: five real datasets (greys) vs v1/v2 default seed 0 (dashed); v3 distributions were not saved by S09")

handles = [plt.Line2D([], [], marker="o", ls="", color=C_REAL, label="real dataset")] + \
          [plt.Line2D([], [], marker="o", ls="", color=C_GEN[g], label=g) for g in ["v1", "v2", "v3"]] + \
          [plt.Rectangle((0, 0), 1, 1, color="#DDDDDD", label="real range")]
fig.legend(handles=handles, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.0), fontsize=6.5)
save(fig, "Fig2")

[save] main/Fig2: 171.4 x 183.1 mm, PNG 4048 x 4326 px @ 600 dpi


(PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig2.pdf'),
 PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig2.png'))

In [6]:
v = prov.values
prov.value("s09_realism_met_by_seed", gates3["realism"]["met_count_by_seed"])
prov.value("s09_unmet_all_seeds", sorted(set.intersection(*[set(x) for x in gates3["realism"]["unmet_by_seed"].values()])))
prov.legend(f"""
**Figure 2 | A benchmark calibrated to six real datasets (S08, S09, S19).**
**a**, Six of the {v['n_real_statistics_ids']} S08 statistics (IDs C1–C8, G1–G5, L1–L2, S1–S4, T2; {cmp3.statistic.nunique()} sub-statistics).
Black dots: value (or median) in each real dataset (scNT-seq haematopoiesis and neurons, LARRY, Setty CD34⁺,
pancreas, dentate gyrus; not every statistic applies to every dataset); grey band: real min–max range. Coloured dots:
one per synthetic dataset (v1: 3 seeds of the default preset; v2 and v3: 3 seeds × each preset). Zero fraction:
real {f2(v['a_C5_zero_fraction_real_min'])}–{f2(v['a_C5_zero_fraction_real_max'])} vs v1 median {f2(v['a_C5_zero_fraction_v1_median'])},
v2 {f2(v['a_C5_zero_fraction_v2_median'])}, v3 {f2(v['a_C5_zero_fraction_v3_median'])}.
**b**, Whether the synthetic median lies in the real range, per statistic and preset (fraction of three seeds); grey, not
applicable. **c**, Fraction of statistics matched under the S08 rule (distance ≤ max(real spread, fixed tolerance) for every
applicable real reference): v1 {f2(v['c_match_rate_v1'])}, v2_default {f2(v['c_match_rate_v2_default'])},
v3_default {f2(v['c_match_rate_v3_default'])}, v3_batch {f2(v['c_match_rate_v3_batch'])}, v3_stress {f2(v['c_match_rate_v3_stress'])},
v3_stationary {f2(v['c_match_rate_v3_stationary'])}; dyngen (S19, nine datasets, bifurcating/cycle/linear × 3 seeds, 25 applicable
statistics) {f2(v['c_match_rate_dyngen_mean'])} (range {f2(v['c_match_rate_dyngen_min'])}–{f2(v['c_match_rate_dyngen_max'])}).
Against the 13 pre-fixed S09 calibration targets, v3_default met 6 of 13 on each seed (realism gate failed);
targets unmet on all three seeds: {', '.join(v['s09_unmet_all_seeds'])}. The user accepted v3 with these misses documented (S09 decision).
**d**, Empirical CDFs of three saved S08 distributions for five real datasets and v1/v2 default seed 0 (v3 distributions
were not saved). Descriptive comparison; the S09 13-target count was pre-registered, the S08/S19 matrices are report only.
""")
prov.write()